### Setup & Imports

Load required libraries (pandas, numpy, pathlib) and define file paths to sample data.
This prepares the environment for working with the Excel files.

In [23]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up paths
SAMPLES_DIR = Path("samples")
MASTER_FILE = SAMPLES_DIR / "Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED.xlsx"
BSNY_CONCUR_FILE = SAMPLES_DIR / "BSNY - SAP & Concur Repoort May 2025 - June 2026.xlsx"
SANCAP_CONCUR_FILE = SAMPLES_DIR / "SanCap - Expense Report USA May 2025 - JUNE 2026.xlsx"


# ------------------------------------------------------------------------------------------
print("✓ Dependencies loaded")
print(f"✓ Sample files directory: {SAMPLES_DIR}")
print(f"✓ Files ready to process")

✓ Dependencies loaded
✓ Sample files directory: samples
✓ Files ready to process


### Prepare source files

In [24]:
# STEP 1: Confirm the source reports exist
print("=" * 80)
print("STEP 1: Confirm Source Reports")
print("=" * 80)

# Check BSNY file
print(f"\n1. BSNY - SAP & Concur Report:")
print(f"   File exists: {BSNY_CONCUR_FILE.exists()}")
if BSNY_CONCUR_FILE.exists():
    bsny_wb = pd.ExcelFile(BSNY_CONCUR_FILE)
    print(f"   Sheets available: {bsny_wb.sheet_names}")
    print(f"   → Has 'Concur' tab: {'Concur' in bsny_wb.sheet_names}")

# Check SANCAP file
print(f"\n2. SanCap - Expense Report USA:")
print(f"   File exists: {SANCAP_CONCUR_FILE.exists()}")
if SANCAP_CONCUR_FILE.exists():
    sancap_wb = pd.ExcelFile(SANCAP_CONCUR_FILE)
    print(f"   Sheets available: {sancap_wb.sheet_names}")

print("\n✓ Source reports confirmed")

STEP 1: Confirm Source Reports

1. BSNY - SAP & Concur Report:
   File exists: True
   Sheets available: ['Concur', 'SAP']
   → Has 'Concur' tab: True

2. SanCap - Expense Report USA:
   File exists: True
   Sheets available: ['page']

✓ Source reports confirmed


### Load Prior-Month Combined Master File

In [25]:
print("\n" + "=" * 80)
print("STEP 2: Load Prior-Month Combined Master File")
print("=" * 80)

print(f"\nLoading: {MASTER_FILE.name}")
master_wb = pd.ExcelFile(MASTER_FILE)
print(f"Sheets in master file: {master_wb.sheet_names}")

# We'll need to reference this for comparison
# Let's load the prior Concur data from the master file
print("\n✓ Prior-month master file loaded")
print("  (Will use this for sanity check in Step 4)")


STEP 2: Load Prior-Month Combined Master File

Loading: Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED.xlsx
Sheets in master file: ['Coding', 'Combined Summary', 'Concur Report', 'SAP Report', 'Uber Report', 'Concur Analysis - PIVOTS', 'SAP Invoices Analysis - PIVOTS', 'Uber Invoices Analysis - PIVOTS', 'All CCs', 'HR HC Combined Jun', 'HR Headcount', 'In Scope CCs']

✓ Prior-month master file loaded
  (Will use this for sanity check in Step 4)


### Apply 2026 Reporting-Period Filter

In [26]:
# STEP 3: Find Year column (should be BA) and filter to 2026
print("\n" + "=" * 80)
print("STEP 3: Apply 2026 Reporting Period Filter")
print("=" * 80)

# Load BSNY Concur tab
print("\n1. BSNY - Concur Tab:")
bsny_concur = pd.read_excel(BSNY_CONCUR_FILE, sheet_name="Concur")
print(f"   Loaded: {bsny_concur.shape[0]} rows × {bsny_concur.shape[1]} columns")

# Find Year column
year_col = None
for col in bsny_concur.columns:
    if 'Year' in str(col) or 'year' in str(col).lower():
        year_col = col
        print(f"   → Found Year column: '{col}'")
        break

if year_col is None:
    # Check if it's column BA (column 53)
    if bsny_concur.shape[1] >= 53:
        year_col = bsny_concur.columns[52]  # BA is column 53 (0-indexed = 52)
        print(f"   → Using column 53 (BA): '{year_col}'")

# Filter to 2026
bsny_concur_2026 = bsny_concur[bsny_concur[year_col] == 2026].copy()
print(f"   → Filtered to Year=2026: {bsny_concur_2026.shape[0]} rows")

# Load SanCap - only 1 sheet, so load directly
print("\n2. SanCap - Expense Report:")
sancap_concur = pd.read_excel(SANCAP_CONCUR_FILE)
print(f"   Loaded: {sancap_concur.shape[0]} rows × {sancap_concur.shape[1]} columns")

sancap_concur_2026 = sancap_concur[sancap_concur[year_col] == 2026].copy()
print(f"   → Filtered to Year=2026: {sancap_concur_2026.shape[0]} rows")

print("\n✓ 2026 data filtered for both entities")

# Display sample data: first 5 and last 5 rows
print("\n" + "=" * 80)
print("SAMPLE DATA - First 5 rows")
print("=" * 80)

# Get first 5 columns + year column
first_cols = list(bsny_concur_2026.columns[:5]) + [year_col]
print("\nBSNY Concur 2026 - FIRST 5 ROWS:")
print(bsny_concur_2026[first_cols].head(5).to_string())

print("\n\nSanCap Concur 2026 - FIRST 5 ROWS:")
print(sancap_concur_2026[first_cols].head(5).to_string())

print("\n" + "=" * 80)
print("SAMPLE DATA - Last 5 rows")
print("=" * 80)

print("\nBSNY Concur 2026 - LAST 5 ROWS:")
print(bsny_concur_2026[first_cols].tail(5).to_string())

print("\n\nSanCap Concur 2026 - LAST 5 ROWS:")
print(sancap_concur_2026[first_cols].tail(5).to_string())


STEP 3: Apply 2026 Reporting Period Filter

1. BSNY - Concur Tab:
   Loaded: 34118 rows × 72 columns
   → Found Year column: 'Year'
   → Filtered to Year=2026: 15377 rows

2. SanCap - Expense Report:
   Loaded: 68915 rows × 72 columns
   → Filtered to Year=2026: 33177 rows

✓ 2026 data filtered for both entities

SAMPLE DATA - First 5 rows

BSNY Concur 2026 - FIRST 5 ROWS:
   Employee First Name Employee Last Name            Employee Login ID  Employee ID       Report Name  Year
13              Gareth             Davies  n100167@SCIBUS.santander.us       100167             April  2026
14              Gareth             Davies  n100167@SCIBUS.santander.us       100167             April  2026
42              Gareth             Davies  n100167@SCIBUS.santander.us       100167     December 2025  2026
43              Gareth             Davies  n100167@SCIBUS.santander.us       100167     December 2025  2026
91              Gareth             Davies  n100167@SCIBUS.santander.us       100167

### Sanity Check: Compare Totals to Prior Month


In [27]:
# STEP 4: Sanity Check - Compare Totals to Prior Month (IN-SCOPE ONLY)
print("\n" + "=" * 80)
print("STEP 4: Sanity Check - Compare Totals to Prior Month (IN-SCOPE ONLY)")
print("=" * 80)

# Find Expense Amount column (should be exactly "Expense Amount (reimbursement currency)")
expense_col = None
for col in bsny_concur.columns:
    if col == "Expense Amount (reimbursement currency)":
        expense_col = col
        print(f"   Found Expense Amount column: '{col}'")
        break

if expense_col is None:
    print(f"   ✗ ERROR: Column 'Expense Amount (reimbursement currency)' not found")
    print(f"   Available columns containing 'amount':")
    for col in bsny_concur.columns:
        if 'amount' in str(col).lower():
            print(f"      - {col}")
else:
    try:
        # Load PRIOR MONTH from master file
        print(f"\n📄 Loading prior-month data from 'Concur Report' sheet...")
        master_concur = pd.read_excel(MASTER_FILE, sheet_name="Concur Report")
        print(f"   Total rows in prior month: {len(master_concur)}")
        
        # Filter PRIOR MONTH to only IN-SCOPE rows using "In Scope" column
        if "In Scope" in master_concur.columns:
            master_in_scope = master_concur[master_concur["In Scope"] == "Yes"]
            prior_total_in_scope = master_in_scope[expense_col].sum()
            
            print(f"\n📊 PRIOR MONTH (IN-SCOPE ONLY):")
            print(f"   Total rows in prior month:       {len(master_concur)}")
            print(f"   Rows marked as 'In Scope: Yes':  {len(master_in_scope)}")
            print(f"   Prior Total (In-Scope):          ${prior_total_in_scope:>15,.2f}")
        else:
            print(f"   ✗ ERROR: 'In Scope' column not found in Concur Report sheet")
            prior_total_in_scope = 0
            master_in_scope = pd.DataFrame()
        
        # CURRENT MONTH: Sum ALL 2026 data (in-scope marking happens in Step 10)
        bsny_current = bsny_concur_2026[expense_col].sum()
        sancap_current = sancap_concur_2026[expense_col].sum()
        combined_current = bsny_current + sancap_current
        
        print(f"\n📊 CURRENT MONTH (2026 - ALL DATA):")
        print(f"   BSNY Concur:     ${bsny_current:>15,.2f}")
        print(f"   SanCap Concur:   ${sancap_current:>15,.2f}")
        print(f"   ─────────────────────────────────")
        print(f"   Combined Total:  ${combined_current:>15,.2f}")
        print(f"\n   Note: New data will be marked as 'In Scope: Yes/No' in Step 10")
        print(f"         (After formulas in 'CC Expense is Mapped to' are applied)")
        
        # SANITY CHECK: Compare current vs prior in-scope
        print(f"\n" + "=" * 80)
        print(f"✓ SANITY CHECK COMPARISON:")
        print(f"=" * 80)
        print(f"   Current Month (all 2026):   ${combined_current:>15,.2f}")
        print(f"   Prior Month (in-scope only): ${prior_total_in_scope:>15,.2f}")
        print(f"   Difference:                  ${combined_current - prior_total_in_scope:>15,.2f}")
        
        if combined_current >= prior_total_in_scope:
            print(f"\n✅ PASS: Current >= Prior (In-Scope)")
            print(f"   YTD totals are increasing correctly ✓")
            print(f"   Data ready for refresh to Step 6 ✓")
        else:
            print(f"\n❌ FAIL: Current < Prior (In-Scope)!")
            print(f"\n   ⚠️  ERROR: The total DECREASED! This should not happen for year-to-date data.")
            print(f"   Action: STOP - Do not continue with refresh!")
            print(f"   Reason: Check if data was correctly filtered to 2026")
            
    except Exception as e:
        print(f"   ✗ ERROR during sanity check: {e}")
        print(f"   Traceback: {type(e).__name__}")


STEP 4: Sanity Check - Compare Totals to Prior Month (IN-SCOPE ONLY)
   Found Expense Amount column: 'Expense Amount (reimbursement currency)'

📄 Loading prior-month data from 'Concur Report' sheet...
   Total rows in prior month: 48554

📊 PRIOR MONTH (IN-SCOPE ONLY):
   Total rows in prior month:       48554
   Rows marked as 'In Scope: Yes':  10585
   Prior Total (In-Scope):          $   2,364,474.49

📊 CURRENT MONTH (2026 - ALL DATA):
   BSNY Concur:     $   3,022,379.23
   SanCap Concur:   $   7,783,763.45
   ─────────────────────────────────
   Combined Total:  $  10,806,142.68

   Note: New data will be marked as 'In Scope: Yes/No' in Step 10
         (After formulas in 'CC Expense is Mapped to' are applied)

✓ SANITY CHECK COMPARISON:
   Current Month (all 2026):   $  10,806,142.68
   Prior Month (in-scope only): $   2,364,474.49
   Difference:                  $   8,441,668.19

✅ PASS: Current >= Prior (In-Scope)
   YTD totals are increasing correctly ✓
   Data ready for refre

### Master File Column Structure Analysis

In [28]:
# STEP 5: Master File Column Structure Analysis
print("=" * 100)
print("STEP 5: Master File Column Structure Analysis")
print("=" * 100)

# Load Master file
master_df = pd.read_excel(MASTER_FILE, sheet_name="Concur Report")

print(f"\n📊 Master File - 'Concur Report' Sheet")
print(f"   Total rows: {len(master_df)}")
print(f"   Total columns: {len(master_df.columns)}")

# Helper columns to identify
helper_columns = ["First & Last Name", "CC Expense is Mapped to", "LOB", "In Scope", "Error"]

# Create detailed column mapping
column_info = []
for idx, col_name in enumerate(master_df.columns):
    # Convert index to column letter
    col_letter = ""
    n = idx
    while n >= 0:
        col_letter = chr(65 + (n % 26)) + col_letter
        n = n // 26 - 1
    
    is_helper = "YES ✓ HELPER" if col_name in helper_columns else "No"
    data_type = str(master_df[col_name].dtype)
    
    column_info.append({
        'Position': col_letter,
        'Index': idx,
        'Column Name': col_name,
        'Type': data_type,
        'Helper?': is_helper
    })

# Display key columns
print("\nKey Columns:")
key_cols_to_show = ["Year", "First Name", "Last Name", "First & Last Name", "CC Expense is Mapped to", 
                     "LOB", "In Scope", "Error"]
for col_info_dict in column_info:
    if col_info_dict['Column Name'] in key_cols_to_show:
        print(f"   {col_info_dict['Position']:>3} | Idx {col_info_dict['Index']:>3} | {col_info_dict['Column Name']:<30} | {col_info_dict['Type']:<10} | {col_info_dict['Helper?']}")

# Identify helper columns
print("\n" + "─" * 100)
print("HELPER COLUMNS (These have formulas, NOT replaced with source data):")
print("─" * 100)
for col_name in helper_columns:
    if col_name in master_df.columns:
        col_idx = list(master_df.columns).index(col_name)
        col_letter = ""
        n = col_idx
        while n >= 0:
            col_letter = chr(65 + (n % 26)) + col_letter
            n = n // 26 - 1
        print(f"   ✓ {col_letter:>3} | {col_name:<30}")
    else:
        print(f"   ✗ NOT FOUND: {col_name}")

# Sample data
print("\n" + "─" * 100)
print("SAMPLE DATA - First 3 rows:")
print("─" * 100)
print(master_df.head(3).to_string())

# Row count by In Scope status
print("\n" + "─" * 100)
print("ROW COUNT SUMMARY:")
print("─" * 100)
print(f"Total rows in Master: {len(master_df)}")
if "In Scope" in master_df.columns:
    scope_counts = master_df["In Scope"].value_counts()
    print(scope_counts)

print("\n✓ Step 5 Complete")

STEP 5: Master File Column Structure Analysis

📊 Master File - 'Concur Report' Sheet
   Total rows: 48554
   Total columns: 77

Key Columns:
     C | Idx   2 | First & Last Name              | str        | YES ✓ HELPER
    BB | Idx  53 | Year                           | int64      | No
    BV | Idx  73 | CC Expense is Mapped to        | str        | YES ✓ HELPER
    BW | Idx  74 | LOB                            | str        | YES ✓ HELPER
    BX | Idx  75 | In Scope                       | str        | YES ✓ HELPER
    BY | Idx  76 | Error                          | str        | YES ✓ HELPER

────────────────────────────────────────────────────────────────────────────────────────────────────
HELPER COLUMNS (These have formulas, NOT replaced with source data):
────────────────────────────────────────────────────────────────────────────────────────────────────
   ✓   C | First & Last Name             
   ✓  BV | CC Expense is Mapped to       
   ✓  BW | LOB                           
   

### BSNY Source File Column Analysis

In [29]:
print("\n" + "=" * 100)
print("STEP 6: BSNY Source File Column Analysis")
print("=" * 100)

# Load BSNY file
bsny_df = pd.read_excel(BSNY_CONCUR_FILE, sheet_name="Concur")

print(f"\n📊 BSNY File - 'Concur' Sheet")
print(f"   Total rows: {len(bsny_df)}")
print(f"   Total columns: {len(bsny_df.columns)}")

# Find Year column
year_col_bsny = None
for col in bsny_df.columns:
    if 'Year' in str(col) or col.lower() == 'year':
        year_col_bsny = col
        break

if year_col_bsny is None and len(bsny_df.columns) > 52:
    year_col_bsny = bsny_df.columns[52]

print(f"\n   Year column found: '{year_col_bsny}'")

# Find Custom columns
print("\n" + "─" * 100)
print("CUSTOM COLUMNS (Mapping targets):")
print("─" * 100)

for col_name in bsny_df.columns:
    if 'Custom 41' in str(col_name):
        col_idx = list(bsny_df.columns).index(col_name)
        col_letter = ""
        n = col_idx
        while n >= 0:
            col_letter = chr(65 + (n % 26)) + col_letter
            n = n // 26 - 1
        print(f"   ✓ Custom 41: '{col_name}' at position {col_letter}")
    elif 'Custom 42' in str(col_name):
        col_idx = list(bsny_df.columns).index(col_name)
        col_letter = ""
        n = col_idx
        while n >= 0:
            col_letter = chr(65 + (n % 26)) + col_letter
            n = n // 26 - 1
        print(f"   ✓ Custom 42: '{col_name}' at position {col_letter}")
    elif 'Custom 43' in str(col_name):
        col_idx = list(bsny_df.columns).index(col_name)
        col_letter = ""
        n = col_idx
        while n >= 0:
            col_letter = chr(65 + (n % 26)) + col_letter
            n = n // 26 - 1
        print(f"   ✓ Custom 43: '{col_name}' at position {col_letter}")

# Row breakdown by Year
print("\n" + "─" * 100)
print("ROW BREAKDOWN BY YEAR:")
print("─" * 100)
year_counts = bsny_df[year_col_bsny].value_counts().sort_index(ascending=False)
print(year_counts)

# Sum expense amounts by Year
if "Expense Amount (reimbursement currency)" in bsny_df.columns:
    print("\n" + "─" * 100)
    print("EXPENSE AMOUNT TOTALS BY YEAR:")
    print("─" * 100)
    expense_by_year = bsny_df.groupby(year_col_bsny)["Expense Amount (reimbursement currency)"].sum().sort_index(ascending=False)
    for year, amount in expense_by_year.items():
        print(f"   {year}: ${amount:>15,.2f}")

# Filter to 2026 for later use
bsny_2026_raw = bsny_df[bsny_df[year_col_bsny] == 2026].copy()
print(f"\n📌 BSNY 2026 Data: {len(bsny_2026_raw)} rows (to be inserted)")

print("\n✓ Step 6 Complete")


STEP 6: BSNY Source File Column Analysis

📊 BSNY File - 'Concur' Sheet
   Total rows: 34118
   Total columns: 72

   Year column found: 'Year'

────────────────────────────────────────────────────────────────────────────────────────────────────
CUSTOM COLUMNS (Mapping targets):
────────────────────────────────────────────────────────────────────────────────────────────────────
   ✓ Custom 41: 'Custom 41 - Name' at position BR
   ✓ Custom 42: 'Custom 42 - Name' at position BS
   ✓ Custom 43: 'Custom 43 - Name' at position BT

────────────────────────────────────────────────────────────────────────────────────────────────────
ROW BREAKDOWN BY YEAR:
────────────────────────────────────────────────────────────────────────────────────────────────────
Year
2026    15377
2025    18741
Name: count, dtype: int64

────────────────────────────────────────────────────────────────────────────────────────────────────
EXPENSE AMOUNT TOTALS BY YEAR:
───────────────────────────────────────────────────

### SanCap Source File Column Analysis

In [30]:
print("\n" + "=" * 100)
print("STEP 7: SanCap Source File Column Analysis")
print("=" * 100)

# Load SanCap file (single sheet)
sancap_df = pd.read_excel(SANCAP_CONCUR_FILE)

print(f"\n📊 SanCap File (single sheet)")
print(f"   Total rows: {len(sancap_df)}")
print(f"   Total columns: {len(sancap_df.columns)}")

# Find Year column
year_col_sancap = None
for col in sancap_df.columns:
    if 'Year' in str(col) or col.lower() == 'year':
        year_col_sancap = col
        break

if year_col_sancap is None and len(sancap_df.columns) > 52:
    year_col_sancap = sancap_df.columns[52]

print(f"\n   Year column found: '{year_col_sancap}'")

# Find Custom columns
print("\n" + "─" * 100)
print("CUSTOM COLUMNS (Mapping targets):")
print("─" * 100)

for col_name in sancap_df.columns:
    if 'Custom 41' in str(col_name):
        col_idx = list(sancap_df.columns).index(col_name)
        col_letter = ""
        n = col_idx
        while n >= 0:
            col_letter = chr(65 + (n % 26)) + col_letter
            n = n // 26 - 1
        print(f"   ✓ Custom 41: '{col_name}' at position {col_letter}")
    elif 'Custom 42' in str(col_name):
        col_idx = list(sancap_df.columns).index(col_name)
        col_letter = ""
        n = col_idx
        while n >= 0:
            col_letter = chr(65 + (n % 26)) + col_letter
            n = n // 26 - 1
        print(f"   ✓ Custom 42: '{col_name}' at position {col_letter}")
    elif 'Custom 43' in str(col_name):
        col_idx = list(sancap_df.columns).index(col_name)
        col_letter = ""
        n = col_idx
        while n >= 0:
            col_letter = chr(65 + (n % 26)) + col_letter
            n = n // 26 - 1
        print(f"   ✓ Custom 43: '{col_name}' at position {col_letter}")

# Row breakdown by Year
print("\n" + "─" * 100)
print("ROW BREAKDOWN BY YEAR:")
print("─" * 100)
year_counts_sancap = sancap_df[year_col_sancap].value_counts().sort_index(ascending=False)
print(year_counts_sancap)

# Sum expense amounts by Year
if "Expense Amount (reimbursement currency)" in sancap_df.columns:
    print("\n" + "─" * 100)
    print("EXPENSE AMOUNT TOTALS BY YEAR:")
    print("─" * 100)
    expense_by_year_sancap = sancap_df.groupby(year_col_sancap)["Expense Amount (reimbursement currency)"].sum().sort_index(ascending=False)
    for year, amount in expense_by_year_sancap.items():
        print(f"   {year}: ${amount:>15,.2f}")

# Filter to 2026 for later use
sancap_2026_raw = sancap_df[sancap_df[year_col_sancap] == 2026].copy()
print(f"\n📌 SanCap 2026 Data: {len(sancap_2026_raw)} rows (to be inserted)")

print("\n✓ Step 7 Complete")


STEP 7: SanCap Source File Column Analysis

📊 SanCap File (single sheet)
   Total rows: 68915
   Total columns: 72

   Year column found: 'Year'

────────────────────────────────────────────────────────────────────────────────────────────────────
CUSTOM COLUMNS (Mapping targets):
────────────────────────────────────────────────────────────────────────────────────────────────────
   ✓ Custom 41: 'Custom 41 - Name' at position BR
   ✓ Custom 42: 'Custom 42 - Name' at position BS
   ✓ Custom 43: 'Custom 43 - Name' at position BT

────────────────────────────────────────────────────────────────────────────────────────────────────
ROW BREAKDOWN BY YEAR:
────────────────────────────────────────────────────────────────────────────────────────────────────
Year
2026    33177
2025    35738
Name: count, dtype: int64

────────────────────────────────────────────────────────────────────────────────────────────────────
EXPENSE AMOUNT TOTALS BY YEAR:
─────────────────────────────────────────────────

### Column Mapping Validation

In [31]:
print("\n" + "=" * 100)
print("STEP 8: Column Mapping Validation")
print("=" * 100)

# Define the 3 required mappings
mappings = [
    {'source': 'Custom 41 - Name', 'target': 'Client Name'},
    {'source': 'Custom 42 - Name', 'target': 'Project Name'},
    {'source': 'Custom 43 - Name', 'target': 'Epense Name'}
]

print("\n" + "─" * 100)
print("MAPPING VALIDATION TABLE:")
print("─" * 100)

all_mappings_valid = True

for mapping in mappings:
    source_col = mapping['source']
    target_col = mapping['target']
    
    # Check source in BSNY
    bsny_source_exists = source_col in bsny_df.columns
    # Check source in SanCap
    sancap_source_exists = source_col in sancap_df.columns
    # Check target in Master
    master_target_exists = target_col in master_df.columns
    
    # Status
    status = "✓ VALID" if (bsny_source_exists and sancap_source_exists and master_target_exists) else "✗ ERROR"
    if status == "✗ ERROR":
        all_mappings_valid = False
    
    print(f"\n{status}: {source_col} → {target_col}")
    print(f"   BSNY Source: {'✓ Found' if bsny_source_exists else '✗ NOT FOUND'}")
    print(f"   SanCap Source: {'✓ Found' if sancap_source_exists else '✗ NOT FOUND'}")
    print(f"   Master Target: {'✓ Found' if master_target_exists else '✗ NOT FOUND'}")

print("\n" + "─" * 100)
if all_mappings_valid:
    print("✓ ALL MAPPINGS VALID - Ready for Step 9")
else:
    print("✗ SOME MAPPINGS INVALID - Check errors above")
print("─" * 100)

print("\n✓ Step 8 Complete")


STEP 8: Column Mapping Validation

────────────────────────────────────────────────────────────────────────────────────────────────────
MAPPING VALIDATION TABLE:
────────────────────────────────────────────────────────────────────────────────────────────────────

✓ VALID: Custom 41 - Name → Client Name
   BSNY Source: ✓ Found
   SanCap Source: ✓ Found
   Master Target: ✓ Found

✓ VALID: Custom 42 - Name → Project Name
   BSNY Source: ✓ Found
   SanCap Source: ✓ Found
   Master Target: ✓ Found

✓ VALID: Custom 43 - Name → Epense Name
   BSNY Source: ✓ Found
   SanCap Source: ✓ Found
   Master Target: ✓ Found

────────────────────────────────────────────────────────────────────────────────────────────────────
✓ ALL MAPPINGS VALID - Ready for Step 9
────────────────────────────────────────────────────────────────────────────────────────────────────

✓ Step 8 Complete


### Row Count & Totals Summary

In [32]:
print("\n" + "=" * 100)
print("STEP 9: Row Count & Totals Summary")
print("=" * 100)

print("\n" + "─" * 100)
print("SUMMARY TABLE:")
print("─" * 100)

# Get metrics
master_total_rows = len(master_df)
bsny_2026_rows = len(bsny_2026_raw)
sancap_2026_rows = len(sancap_2026_raw)
combined_2026_rows = bsny_2026_rows + sancap_2026_rows

# Calculate expected final row count
expected_final_rows = 1 + combined_2026_rows  # 1 for header

# Expense totals
bsny_2026_expense = bsny_2026_raw["Expense Amount (reimbursement currency)"].sum() if "Expense Amount (reimbursement currency)" in bsny_2026_raw.columns else 0
sancap_2026_expense = sancap_2026_raw["Expense Amount (reimbursement currency)"].sum() if "Expense Amount (reimbursement currency)" in sancap_2026_raw.columns else 0
combined_2026_expense = bsny_2026_expense + sancap_2026_expense

# Print summary
summary_data = {
    'Metric': [
        'Master Current Rows (total)',
        'BSNY 2026 Rows (new)',
        'SanCap 2026 Rows (new)',
        'Combined 2026 Rows',
        '',
        'Expected Final Rows (after refresh)',
        '',
        'BSNY 2026 Expense Total',
        'SanCap 2026 Expense Total',
        'Combined 2026 Expense Total'
    ],
    'Value': [
        f"{master_total_rows:,}",
        f"{bsny_2026_rows:,}",
        f"{sancap_2026_rows:,}",
        f"{combined_2026_rows:,}",
        '',
        f"{expected_final_rows:,}",
        '',
        f"${bsny_2026_expense:,.2f}",
        f"${sancap_2026_expense:,.2f}",
        f"${combined_2026_expense:,.2f}"
    ]
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

print("\n" + "─" * 100)
print("VALIDATION CHECKS:")
print("─" * 100)
print(f"✓ Master file has data: {master_total_rows > 0}")
print(f"✓ BSNY 2026 data exists: {bsny_2026_rows > 0}")
print(f"✓ SanCap 2026 data exists: {sancap_2026_rows > 0}")
print(f"✓ Combined data ready: {combined_2026_rows > 0}")
print(f"✓ All mappings valid: {all_mappings_valid}")

print("\n✓ Step 9 Complete - Ready for Step 10!")
print("\n📌 Next: Step 10 will create outputs folder, copy Master, and insert new data with formulas")


STEP 9: Row Count & Totals Summary

────────────────────────────────────────────────────────────────────────────────────────────────────
SUMMARY TABLE:
────────────────────────────────────────────────────────────────────────────────────────────────────
                             Metric          Value
        Master Current Rows (total)         48,554
               BSNY 2026 Rows (new)         15,377
             SanCap 2026 Rows (new)         33,177
                 Combined 2026 Rows         48,554
                                                  
Expected Final Rows (after refresh)         48,555
                                                  
            BSNY 2026 Expense Total  $3,022,379.23
          SanCap 2026 Expense Total  $7,783,763.45
        Combined 2026 Expense Total $10,806,142.68

────────────────────────────────────────────────────────────────────────────────────────────────────
VALIDATION CHECKS:
────────────────────────────────────────────────────────────────

### DEBUG: Inspect 2D Array Structure

In [38]:
# DEBUG: Check array structure and data
print("\n" + "=" * 100)
print("DEBUG: Inspect Data Array Structure")
print("=" * 100)

# Show combined data structure
print(f"\nCombined data shape: {combined_data.shape}")
print(f"Columns: {len(combined_data.columns)}")
print(f"Rows: {len(combined_data)}")

# Get master column names (excluding helpers)
helper_columns = ["First & Last Name", "CC Expense is Mapped to", "LOB", "In Scope", "Error"]
master_df_reload = pd.read_excel(MASTER_FILE, sheet_name="Concur Report", nrows=1)
non_helper_cols = [col for col in master_df_reload.columns if col not in helper_columns]

print(f"\nMaster file non-helper columns: {len(non_helper_cols)}")
print(f"  First 10: {list(non_helper_cols)[:10]}")
print(f"  Last 5: {list(non_helper_cols)[-5:]}")

# Check if we can map each master column to source
print(f"\nColumn mapping validation (sample):")
mapping_issues = []
for master_col in list(non_helper_cols)[:10]:
    # Determine source column name
    source_col = master_col
    if master_col == 'Client Name':
        source_col = 'Custom 41 - Name'
    elif master_col == 'Project Name':
        source_col = 'Custom 42 - Name'
    elif master_col == 'Epense Name':
        source_col = 'Custom 43 - Name'
    
    # Check if source exists in combined data
    if source_col in combined_data.columns:
        sample_val = combined_data[source_col].iloc[0]
        print(f"  ✓ {master_col:30} → {source_col:30} = {sample_val}")
    else:
        print(f"  ✗ {master_col:30} → {source_col:30} NOT FOUND")
        mapping_issues.append((master_col, source_col))

if mapping_issues:
    print(f"\n⚠️  Found {len(mapping_issues)} mapping issues. Checking column names...")
    print(f"\nActual columns in combined_data:")
    for i, col in enumerate(combined_data.columns):
        print(f"  {i}: {col}")

print("\n✓ Debug inspection complete")



DEBUG: Inspect Data Array Structure

Combined data shape: (48554, 72)
Columns: 72
Rows: 48554

Master file non-helper columns: 72
  First 10: ['Employee First Name', 'Employee Last Name', 'Employee Login ID', 'Employee ID', 'Report Name', 'Custom 5 - Code', 'Custom 5 - Name', 'From Location', 'To Location', 'Business Distance']
  Last 5: ['Custom 16 - Name', 'Custom 20 - Name', 'Client Name', 'Project Name', 'Epense Name']

Column mapping validation (sample):
  ✓ Employee First Name            → Employee First Name            = Gareth
  ✓ Employee Last Name             → Employee Last Name             = Davies
  ✓ Employee Login ID              → Employee Login ID              = n100167@SCIBUS.santander.us
  ✓ Employee ID                    → Employee ID                    = 100167
  ✓ Report Name                    → Report Name                    = April
  ✓ Custom 5 - Code                → Custom 5 - Code                = BTC
  ✓ Custom 5 - Name                → Custom 5 - Name    

## Data Refresh (Step 10 = 3.1.2):

### DEBUG: Simulate 2D Array Building for COM

In [ ]:
# DEBUG: Simulate exactly what Step 10.5 does - build the 2D array
print("\n" + "=" * 100)
print("DEBUG: Simulate 2D Array Building (Step 10.5 logic)")
print("=" * 100)

# Reload master to get column names
master_df_for_cols = pd.read_excel(MASTER_FILE, sheet_name="Concur Report", nrows=1)
helper_columns = ["First & Last Name", "CC Expense is Mapped to", "LOB", "In Scope", "Error"]

# Get non-helper columns in the same order as the Master file
master_cols_ordered = [col for col in master_df_for_cols.columns if col not in helper_columns]

print(f"\nBuilding array with:")
print(f"  - {len(master_cols_ordered)} Master columns")
print(f"  - {len(combined_data)} data rows")
print(f"  - Array dimensions: {len(combined_data)} × {len(master_cols_ordered)}")

# Build TEST 2D array (just first 5 rows for inspection)
test_array = []
for idx in range(min(5, len(combined_data))):
    row_data = combined_data.iloc[idx]
    row_values = []
    
    for master_col_name in master_cols_ordered:
        # Apply mapping
        source_col_name = master_col_name
        if master_col_name == 'Client Name':
            source_col_name = 'Custom 41 - Name'
        elif master_col_name == 'Project Name':
            source_col_name = 'Custom 42 - Name'
        elif master_col_name == 'Epense Name':
            source_col_name = 'Custom 43 - Name'
        
        # Get value
        if source_col_name in row_data.index:
            value = row_data[source_col_name]
            native_value = convert_value_to_native(value)
            row_values.append(native_value)
        else:
            row_values.append(None)
    
    test_array.append(row_values)

# Display test array
print(f"\nTest array (first 5 rows × first 5 columns):")
print(f"  Row 0: {test_array[0][:5]}")
print(f"  Row 1: {test_array[1][:5]}")
print(f"  Row 2: {test_array[2][:5]}")

# Check types
print(f"\nData types in first row:")
for i, val in enumerate(test_array[0][:5]):
    print(f"  Col {i}: {type(val).__name__} = {val}")

# Check last columns (Client Name, Project Name, Epense Name)
print(f"\nTest array (first 5 rows × LAST 3 columns - Custom 41/42/43 mapping):")
print(f"  Client Name, Project Name, Epense Name columns:")
for row_idx, row in enumerate(test_array):
    print(f"  Row {row_idx}: {row[-3:]}")

print("\n✓ Array simulation complete - ready for full COM operation")


### Create Outputs Folder & Copy Master File

In [42]:
print("\n" + "=" * 100)
print("STEP 10: Data Refresh - Clear Master & Insert New Data")
print("=" * 100)

import shutil
from pathlib import Path
import os

# Step 10.1: Create outputs folder and copy Master file
print("\n" + "─" * 100)
print("STEP 10.1: Create Outputs Folder & Copy Master File")
print("─" * 100)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_MASTER_FILE = OUTPUT_DIR / MASTER_FILE.name

print(f"\nCopying Master file to outputs/...")
shutil.copy(MASTER_FILE, OUTPUT_MASTER_FILE)
print(f"✓ Copied to: {OUTPUT_MASTER_FILE}")

# Convert to absolute path string for COM
abs_path = os.path.abspath(str(OUTPUT_MASTER_FILE))
print(f"✓ Absolute path: {abs_path}")

# Helper function to convert pandas values to native Python types
def convert_value_to_native(val):
    """Convert pandas/numpy types to native Python types compatible with COM"""
    import numpy as np
    import pandas as pd
    from datetime import datetime, date
    
    if val is None or pd.isna(val):
        return None
    elif isinstance(val, (pd.Timestamp, datetime)):
        return val.strftime('%Y-%m-%d %H:%M:%S') if hasattr(val, 'strftime') else str(val)
    elif isinstance(val, date):
        return val.isoformat()
    elif isinstance(val, np.integer):
        return int(val)
    elif isinstance(val, np.floating):
        return float(val)
    elif isinstance(val, np.bool_):
        return bool(val)
    elif isinstance(val, (np.ndarray, list)):
        return str(val)
    else:
        return val

# Main COM operation - all in one try-except
try:
    import win32com.client
    
    print(f"\n✓ Opening Excel application...")
    # Open Excel application
    excel_app = win32com.client.Dispatch("Excel.Application")
    excel_app.Visible = False
    excel_app.DisplayAlerts = False  # Suppress alerts
    excel_app.ScreenUpdating = False  # Disable screen updates for speed
    
    print(f"✓ Opening workbook: {abs_path}")
    # Open the copied master file with explicit parameters
    workbook = excel_app.Workbooks.Open(
        Filename=abs_path,
        UpdateLinks=False,
        ReadOnly=False,
        Format=None,
        Password="",
        WriteResPassword="",
        IgnoreReadOnlyRecommended=True,
        Origin=None,
        Delimiter=None,
        Editable=True,
        Notify=False,
        Converter=None,
        AddToMru=False,
        Local=False,
        CorruptLoad=False
    )
    
    if workbook is None:
        raise Exception("Failed to open workbook - Workbooks.Open returned None")
    
    print(f"✓ Opened workbook with COM")
    
    worksheet = workbook.Sheets("Concur Report")
    print(f"✓ Working on sheet: 'Concur Report'")
    
    # ==================================================================================
    # STEP 10.2: Clear Data Rows (Keep Formulas in Helper Columns)
    # ==================================================================================
    print("\n" + "─" * 100)
    print("STEP 10.2: Clear Data Rows (Keep Formulas in Helper Columns)")
    print("─" * 100)
    
    # Find helper columns indices
    helper_columns = ["First & Last Name", "CC Expense is Mapped to", "LOB", "In Scope", "Error"]
    helper_col_indices = {}
    
    for col_idx in range(1, worksheet.UsedRange.Columns.Count + 1):
        col_name = worksheet.Cells(1, col_idx).Value
        if col_name in helper_columns:
            helper_col_indices[col_name] = col_idx
            print(f"   Found helper column '{col_name}' at column {col_idx}")
    
    # Find last row with data
    last_row = worksheet.UsedRange.Rows.Count
    print(f"\n   Current last row: {last_row}")
    
    # Delete all data rows (starting from row 2), keeping header and formulas
    if last_row > 1:
        delete_range = worksheet.Range(f"2:{last_row}")
        delete_range.Delete()
        print(f"   ✓ Deleted rows 2 to {last_row}")
    
    print(f"\n✓ Master file cleared, formulas in helper columns preserved")
    
    # ==================================================================================
    # STEP 10.3: Map Columns & Identify Columns to Transfer
    # ==================================================================================
    print("\n" + "─" * 100)
    print("STEP 10.3: Map Columns & Identify Columns to Transfer")
    print("─" * 100)
    
    # Define column mappings (Master → Source)
    column_mappings = {
        'Client Name': 'Custom 41 - Name',
        'Project Name': 'Custom 42 - Name',
        'Epense Name': 'Custom 43 - Name'
    }
    
    print(f"\nColumn mappings (Master → Source):")
    for master_col, source_col in column_mappings.items():
        print(f"   {master_col:20} ← {source_col}")
    
    # Find all non-helper columns in master
    print(f"\n\nIdentifying all Master columns to transfer...")
    master_columns_to_transfer = []
    for col_idx in range(1, worksheet.UsedRange.Columns.Count + 1):
        col_name = worksheet.Cells(1, col_idx).Value
        if col_name and col_name not in helper_columns:
            master_columns_to_transfer.append((col_idx, col_name))
    
    print(f"✓ Found {len(master_columns_to_transfer)} non-helper columns to process")
    
    # ==================================================================================
    # STEP 10.4: Prepare Data from Source Files
    # ==================================================================================
    print("\n" + "─" * 100)
    print("STEP 10.4: Prepare Data from Source Files")
    print("─" * 100)
    
    # For BSNY
    print(f"\nProcessing BSNY 2026 data ({len(bsny_2026_raw)} rows)...")
    bsny_data_to_insert = bsny_2026_raw.copy()
    
    # For SanCap
    print(f"Processing SanCap 2026 data ({len(sancap_2026_raw)} rows)...")
    sancap_data_to_insert = sancap_2026_raw.copy()
    
    # Combine both datasets
    combined_data = pd.concat([bsny_data_to_insert, sancap_data_to_insert], ignore_index=True)
    print(f"\n✓ Combined data: {len(combined_data)} rows total")
    
    # ==================================================================================
    # STEP 10.5: Build Column Arrays and Insert (Optimized - Column by Column)
    # ==================================================================================
    print("\n" + "─" * 100)
    print("STEP 10.5: Build Column Arrays and Insert (Column-by-Column)")
    print("─" * 100)
    
    start_row = 2
    print(f"\nBuilding {len(master_columns_to_transfer)} column arrays for {len(combined_data)} rows...")
    
    # Build all column arrays
    column_arrays = []
    for col_idx, master_col_name in master_columns_to_transfer:
        # Get source column name (apply mapping if needed)
        source_col_name = master_col_name
        
        if master_col_name == 'Client Name':
            source_col_name = 'Custom 41 - Name'
        elif master_col_name == 'Project Name':
            source_col_name = 'Custom 42 - Name'
        elif master_col_name == 'Epense Name':
            source_col_name = 'Custom 43 - Name'
        
        # Build array for this column
        col_array = []
        for idx, row_data in combined_data.iterrows():
            if source_col_name in row_data.index:
                value = row_data[source_col_name]
                native_value = convert_value_to_native(value)
                col_array.append(native_value)
            else:
                col_array.append(None)
        
        column_arrays.append((col_idx, col_array))
    
    print(f"✓ Built {len(column_arrays)} column arrays")
    
    # Now insert each column array into Excel
    print(f"\nInserting {len(column_arrays)} columns into worksheet...")
    end_row = start_row + len(combined_data) - 1
    
    for col_position, (col_idx, col_array) in enumerate(column_arrays):
        # Create range for this column: e.g., A2:A48555
        col_range = worksheet.Range(
            worksheet.Cells(start_row, col_idx),
            worksheet.Cells(end_row, col_idx)
        )
        # Assign the column array
        col_range.Value = [[val] for val in col_array]  # Convert to 2D array format (n×1)
        
        if (col_position + 1) % 10 == 0:
            print(f"   Inserted {col_position + 1}/{len(column_arrays)} columns...")
    
    print(f"✓ Inserted all {len(combined_data)} rows × {len(master_columns_to_transfer)} columns")
    
    # ==================================================================================
    # STEP 10.6: Extend Helper Column Formulas
    # ==================================================================================
    print("\n" + "─" * 100)
    print("STEP 10.6: Extend Helper Column Formulas")
    print("─" * 100)
    
    new_last_row = end_row  # Last row with data
    
    for helper_col_name, col_idx in helper_col_indices.items():
        # Get formula from row 2
        formula_cell = worksheet.Cells(2, col_idx)
        
        if formula_cell.Value and str(formula_cell.Value).startswith('='):
            print(f"\n   Extending formula in column '{helper_col_name}' (Col {col_idx})")
            # Copy formula down to last row using AutoFill
            source_range = worksheet.Cells(2, col_idx)
            target_range = worksheet.Range(
                worksheet.Cells(2, col_idx),
                worksheet.Cells(new_last_row, col_idx)
            )
            source_range.AutoFill(target_range)
            print(f"      ✓ Formula extended to row {new_last_row}")
    
    # ==================================================================================
    # STEP 10.7: Save and Close
    # ==================================================================================
    print("\n" + "─" * 100)
    print("STEP 10.7: Save and Close")
    print("─" * 100)
    
    # Re-enable screen updates before saving
    print(f"\nRe-enabling screen updates...")
    excel_app.ScreenUpdating = True
    
    print(f"Saving workbook...")
    workbook.Save()
    print(f"✓ Workbook saved successfully")
    
    # Close workbook and Excel with proper COM cleanup
    print(f"\nClosing workbook and Excel...")
    try:
        # Close without saving (already saved above)
        workbook.Close(SaveChanges=False)
        
        # Quit Excel
        excel_app.Quit()
        
        # Force release of COM references
        del workbook
        del excel_app
        import gc
        gc.collect()  # Force garbage collection
        
        # Small delay to ensure file is fully released
        import time
        time.sleep(0.5)
        
        print(f"✓ Excel closed and COM references released")
    except Exception as close_error:
        print(f"   Warning during cleanup: {close_error}")
    
    print(f"\n✓ Step 10 Complete - Data refresh successful!")
    print(f"   Output file: {OUTPUT_MASTER_FILE}")
    print(f"   Total rows inserted: {len(combined_data)}")
    print(f"   Data rows: {start_row}-{end_row}")
    print(f"   Helper columns formulas extended to row: {new_last_row}")

except ImportError:
    print("\n✗ ERROR: win32com not installed")
    print("   Install with: pip install pywin32")
except Exception as e:
    print(f"\n✗ ERROR during COM operation: {e}")
    import traceback
    traceback.print_exc()
    if 'excel_app' in locals():
        try:
            excel_app.ScreenUpdating = True
            excel_app.Quit()
        except:
            pass



STEP 10: Data Refresh - Clear Master & Insert New Data

────────────────────────────────────────────────────────────────────────────────────────────────────
STEP 10.1: Create Outputs Folder & Copy Master File
────────────────────────────────────────────────────────────────────────────────────────────────────

Copying Master file to outputs/...
✓ Copied to: outputs\Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED.xlsx
✓ Absolute path: c:\Users\safuente\OneDrive - Capgemini\Documents\Projects\Automation 28.07.26\deal expenses automation\deal-expense-automation\outputs\Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED.xlsx

✓ Opening Excel application...
✓ Opening workbook: c:\Users\safuente\OneDrive - Capgemini\Documents\Projects\Automation 28.07.26\deal expenses automation\deal-expense-automation\outputs\Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED.xlsx
✓ Opened workbook with COM
✓ Working on sheet: 'Concur Report'

─────────────

In [37]:
# DEBUG: Verify column mapping before COM operation
print("\n" + "=" * 100)
print("DEBUG: Column Mapping Verification")
print("=" * 100)

print("\n1. MASTER FILE COLUMNS (first 10):")
master_test = pd.read_excel(MASTER_FILE, sheet_name="Concur Report", nrows=1)
for idx, col in enumerate(list(master_test.columns)[:10]):
    print(f"   {idx}: {col}")

print("\n2. BSNY SOURCE COLUMNS (first 10):")
for idx, col in enumerate(list(bsny_2026_raw.columns)[:10]):
    print(f"   {idx}: {col}")

print("\n3. SANCAP SOURCE COLUMNS (first 10):")
for idx, col in enumerate(list(sancap_2026_raw.columns)[:10]):
    print(f"   {idx}: {col}")

# Check if Custom/Client/Project/Epense columns exist
print("\n4. CHECKING FOR CUSTOM/MAPPING COLUMNS:")
target_cols = ['Custom 41 - Name', 'Custom 42 - Name', 'Custom 43 - Name', 'Client Name', 'Project Name', 'Epense Name']
for col_name in target_cols:
    in_master = col_name in master_test.columns
    in_bsny = col_name in bsny_2026_raw.columns
    in_sancap = col_name in sancap_2026_raw.columns
    print(f"   {col_name:25} | Master: {str(in_master):5} | BSNY: {str(in_bsny):5} | SanCap: {str(in_sancap):5}")

# Show sample row data
print("\n5. SAMPLE BSNY DATA (first row, columns 0-5):")
if len(bsny_2026_raw) > 0:
    sample_row = bsny_2026_raw.iloc[0]
    for col_name in list(bsny_2026_raw.columns)[:5]:
        print(f"   {col_name}: {sample_row[col_name]}")

print("\n✓ Debug verification complete")



DEBUG: Column Mapping Verification

1. MASTER FILE COLUMNS (first 10):
   0: Employee First Name
   1: Employee Last Name
   2: First & Last Name
   3: Employee Login ID
   4: Employee ID
   5: Report Name
   6: Custom 5 - Code
   7: Custom 5 - Name
   8: From Location
   9: To Location

2. BSNY SOURCE COLUMNS (first 10):
   0: Employee First Name
   1: Employee Last Name
   2: Employee Login ID
   3: Employee ID
   4: Report Name
   5: Custom 5 - Code
   6: Custom 5 - Name
   7: From Location
   8: To Location
   9: Business Distance

3. SANCAP SOURCE COLUMNS (first 10):
   0: Employee First Name
   1: Employee Last Name
   2: Employee Login ID
   3: Employee ID
   4: Report Name
   5: Custom 5 - Code
   6: Custom 5 - Name
   7: From Location
   8: To Location
   9: Business Distance

4. CHECKING FOR CUSTOM/MAPPING COLUMNS:
   Custom 41 - Name          | Master: False | BSNY: True  | SanCap: True 
   Custom 42 - Name          | Master: False | BSNY: True  | SanCap: True 
   Custom 43